In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
import matplotlib.pyplot as plt
# seaborn is a better tool for this with more examples 
import seaborn as sns


df_melanoma = pd.read_csv('melanoma_mitoses.csv', sep=';')

In [2]:
df_melanoma.head()

,ID_prog,age,sex,year,histology,site,pT,pN,pM,TNM_stage,breslow,clark,ulcerazione,regressione,til,growth,positive_slnb,mitoses,status,survival
0,1,31,F,2015,Nodular melanoma,Trunk,pT4,N3c,M0,III,>=4,IV,Present,Absent,Absent,Vertical,0,3,0,648
1,2,53,M,2017,Superficial spreading melanoma,Trunk,pT1,N0,M0,I,< 0.76,II,Absent,Present,Present,Vertical,0,0,0,1147
2,3,67,F,2015,Superficial spreading melanoma,Upper limb,pT1,N0,M0,I,0.76-1.50,IV,Absent,Absent,Present,Vertical,0,0,0,1948
3,4,29,F,2015,Superficial spreading melanoma,Trunk,pT1,N1a,M0,III,< 0.76,III,Absent,Absent,Present,Vertical,1,0,0,2170
4,5,57,F,2015,Superficial spreading melanoma,Lower limb,pT1,N0,M0,I,< 0.76,II,Absent,Absent,Present,Radial,0,0,0,1960


In [3]:
df_melanoma['status'].unique()

array([0, 1])

In [4]:
df_melanoma.shape

(2255, 20)

In [5]:
df_melanoma.columns

Index(['ID_prog', 'age', 'sex', 'year', 'histology', 'site', 'pT', 'pN', 'pM',
       'TNM_stage', 'breslow', 'clark', 'ulcerazione', 'regressione', 'til',
       'growth', 'positive_slnb', 'mitoses', 'status', 'survival'],
      dtype='object')

In [6]:
# Now start to clean the data
# In order to make this model work we will need to encode
# many of the variables to make them numeric

# First we can drop unnecessary columns

df_melanoma_X = df_melanoma.drop(columns=[ 'status',    # column we are predicting
                                           'survival',   # leakage: another way of showing status the variable we are trying to predict
                                           'ID_prog',    # row identifier, no predictive value
                                           'year'])      # not needed

# ORDINAL ENCODINGS 
# This for the variables where we care about the order
# so e.g. the larger the breslow_map (thicker the tumor) 
# the more dangerous it is, so is encoded as 5

tnm_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
df_melanoma_X['TNM_stage'] = df_melanoma_X['TNM_stage'].map(tnm_map)

pt_map = {'pT1': 1, 'pT2': 2, 'pT3': 3, 'pT4': 4}
df_melanoma_X['pT'] = df_melanoma_X['pT'].map(pt_map)

breslow_map = {'< 0.76': 1, '0.76-1.50': 2, '1.51-3.99': 3, '>=4': 4}
df_melanoma_X['breslow'] = df_melanoma_X['breslow'].map(breslow_map)

clark_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5}
df_melanoma_X['clark'] = df_melanoma_X['clark'].map(clark_map)

# BINARY ENCODINGS
# Simple categorization between 2 options

df_melanoma_X['sex']         = df_melanoma_X['sex'].map({'F': 0, 'M': 1})
df_melanoma_X['ulcerazione'] = df_melanoma_X['ulcerazione'].map({'Absent': 0, 'Present': 1})
df_melanoma_X['til']         = df_melanoma_X['til'].map({'Absent': 0, 'Present': 1})
df_melanoma_X['growth']      = df_melanoma_X['growth'].map({'Radial': 0, 'Vertical': 1})
df_melanoma_X['regressione'] = df_melanoma_X['regressione'].map({'Absent': 0, 'Present': 1})
df_melanoma_X['pM']          = df_melanoma_X['pM'].map({'M0': 0, 'M1': 1})
df_melanoma_X['positive_slnb'] = df_melanoma_X['positive_slnb'].map({'0': 0, '1': 1, '>1': 2})

# ORDINAL pN 
pn_map = {'N0': 0, 'N1a': 1, 'N1b': 1, 'N1c': 1,
           'N2a': 2, 'N2b': 2, 'N2c': 2,
           'N3':  3, 'N3c': 3}
df_melanoma_X['pN'] = df_melanoma_X['pN'].map(pn_map)

# ONE-HOT ENCODE 
# Columns we need to encode but don't care about the order
# This is a standard method of working with this kind of data
# drop_first=True drops one dummy per group to avoid multicollinearity
df_melanoma_X = pd.get_dummies(df_melanoma_X, columns=['histology', 'site'], drop_first=True)

# LOG-TRANSFORM 
# This is the recommended way where you have a wide range of values 
# mitosis runs from 0 to 55 and is very skewed to the lower end of 
# that range
df_melanoma_X['mitoses_log'] = np.log1p(df_melanoma_X['mitoses'])

# ... and drop the original mitosis column
df_melanoma_X = df_melanoma_X.drop(columns=['mitoses'])

In [7]:
df_melanoma_X.columns

Index(['age', 'sex', 'pT', 'pN', 'pM', 'TNM_stage', 'breslow', 'clark',
       'ulcerazione', 'regressione', 'til', 'growth', 'positive_slnb',
       'histology_Desmoplastic melanoma', 'histology_Lentigo maligna',
       'histology_Malignant melanoma', 'histology_Nodular melanoma',
       'histology_Spitzoid melanoma',
       'histology_Superficial spreading melanoma', 'site_Head',
       'site_Lower limb', 'site_Trunk', 'site_Upper limb', 'mitoses_log'],
      dtype='object')

In [8]:
df_melanoma_X.head()

,age,sex,pT,pN,pM,TNM_stage,breslow,clark,ulcerazione,regressione,...,histology_Lentigo maligna,histology_Malignant melanoma,histology_Nodular melanoma,histology_Spitzoid melanoma,histology_Superficial spreading melanoma,site_Head,site_Lower limb,site_Trunk,site_Upper limb,mitoses_log
0,31,0,4.0,3.0,0.0,3.0,4.0,4.0,1.0,0.0,...,False,False,True,False,False,False,False,True,False,1.386294
1,53,1,1.0,0.0,0.0,1.0,1.0,2.0,0.0,1.0,...,False,False,False,False,True,False,False,True,False,0.000000
2,67,0,1.0,0.0,0.0,1.0,2.0,4.0,0.0,0.0,...,False,False,False,False,True,False,False,False,True,0.000000
3,29,0,1.0,1.0,0.0,3.0,1.0,3.0,0.0,0.0,...,False,False,False,False,True,False,False,True,False,0.000000
4,57,0,1.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,...,False,False,False,False,True,False,True,False,False,0.000000


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Prepare data
# X here is the training data which must be a table, so 2D. Here is a good explaination..
#The 1D vs 2D distinction is purely about shape:
#  - A Series [1, 2, 3] has shape (3,) — 1D
#  - A DataFrame with one column has shape (3, 1) — 2D
#  - A DataFrame with two columns has shape (3, 2) — 2D

#  So in your case, Season is not the row identifier — it's a feature just like Operator. The row index is separate and
#  implicit. sklearn doesn't use it at all; it just sees a table of values where each row is one observation and each
#  column is one feature.
#  so we could have X = df_new[['Season', 'Operator', 'Locomotive'.....]] we are just creating a table.

X = df_melanoma_X

# only if you need to encode your categories
# ski-kitlearn can't cope with catagories such as "spring" so we have to encode them into 
# expand your 2-column X into ~27 binary columns (4 seasons + 24 operators — minus one each to avoid
# redundancy, a concept called the dummy variable trap). get_dummies handles this with drop_first=True:
# X_encoded = pd.get_dummies(X, columns=['Season', 'Operator'], drop_first=True)


# for Y that must be a series (aka a list) which is expected for target variable.
y = df_melanoma['status']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 1804
Test set size: 451


In [10]:
# Fit logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# View learned parameters
print("Learned parameters:")
print(f"  Intercept (β₀): {model.intercept_[0]:.4f}")
print(f"  Coefficient (β₁): {model.coef_[0][0]:.4f}")
print(f"\nDecision boundary: {-model.intercept_[0]/model.coef_[0][0]:.2f}")

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Late', 'Ontime']))